In [1]:
#RISE settings
from traitlets.config.manager import BaseJSONConfigManager
from pathlib import Path
path = Path.home() / ".jupyter" / "nbconfig"
cm = BaseJSONConfigManager(config_dir=str(path))
tmp = cm.update(
        "rise",
        {
            "theme": "serif",
            "transition": "fade",
            "start_slideshow_at": "selected",
            "autolaunch": False,
            "width": "100%",
            "height": "100%",
            "header": "",
            "footer":"",
            "scroll": True,
            "enable_chalkboard": False,
            "slideNumber": True,
            "center": False,
            "controlsLayout": "edges",
            "slideNumber": True,
            "hash": True,
        }
    )

### Programming CAD for 3D Printed Structured Materials

<table width="75%"><tr>
  <td>
      <img src="imgs/openSCAD_ball_and_stick.png" width="100%">
      <center>
        <font size="5">&quot;Ball and stick&quot; primative cubic lattice model in OpenSCAD</font>
      </center>
  </td> 
  <td>
      <img src="imgs/voxel_electrode_model.png" width="100%">
      <center>
        <font size="5">Voxelated gyroidal surface electrode support model in Python using VoxelCAD and Pyvista</font>
      </center>
  </td> 
  <td>
      <img src="imgs/Ag_gyroids.jpg" width="100%">
      <center>
        <font size="5">3D printed, silver coated gyroidal disks</font>
      </center>
  </td> 
</tr></table>

Materials that are strong, lightweight, and have open continous porosity (no dead-ends/ "blind holes") have many interesting applications:
- light-weight strong construction materials/composites
- liquid absorbant materials (sponges)
- particle size selection filters
- shock absorbers/padding
- sound absorbing/filtering materials
- catalyst supports for chemistry
- electrode supports for sensors (EEG, conductivity, electrochemical, gas concentration...)
- electrodes for devices (batteries, super-capacitors, fuel-cells)
- frameworks for growing artificial organs

### Programming CAD for 3D Printed Structured Materials

<table width="75%"><tr>
  <td>
      <img src="imgs/openSCAD_ball_and_stick.png" width="100%">
      <center>
        <font size="5">&quot;Ball and stick&quot; primative cubic lattice model in OpenSCAD</font>
      </center>
  </td> 
  <td>
      <img src="imgs/voxel_electrode_model.png" width="100%">
      <center>
        <font size="5">Voxelated gyroidal surface electrode support model in Python using VoxelCAD and Pyvista</font>
      </center>
  </td> 
  <td>
      <img src="imgs/Ag_gyroids.jpg" width="100%">
      <center>
        <font size="5">3D printed, silver coated gyroidal disks</font>
      </center>
  </td> 
</tr></table>


- How can we attempt to design parts that have finely detailed, repetitive structure?

- Are existing design and fabrication tools up to the task? 

Spoiler Alert...

- VoxelCAD is a Python package that I am developing to make 3D printable parts that have embedded complex structure
  - user interface is just Python code that leverages *object-oriented design* and *operator overloading* for expressivity (inspired by the elegant [OpenSCAD language](https://openscad.org/))
  - visualizations and mesh data processing will be provided using the [PyVista package](https://docs.pyvista.org/), a very awesome 3rd party project

### 3D Printing
<table width="50%"><tr>
  <td>
    <figure>
      <img src="imgs/prusa_mk3.jpg" width="100%">
      <figcaption><center>Prusa MK3 FDM Printer</center></figcaption>
    </figure>
  </td> 
  <td>
    <figure>
      <img src="imgs/form3.jpg" width="60%">
      <figcaption><center>Formlabs Form3 SLA Printer</center></figcaption>
    </figure>
  </td> 
</tr></table>

A distinct advantage of 3D Printing (additive manufacturing in general) is the relative ease for fabrication of complex structured materials, layer-by-layer.

Consider 2 popular types found in hobby, small business, and academic research contexts:

- Fused deposition modeling (FDM): extrudes a melted plastic filament on a plate
  - good for building large, low resolution, durable parts (ABS plastic)
  - can be most affordable option, open source options, interchangable parts

- Sterolithography (SLA): selectively cures photopolymer resin, pulling solid parts out of the liquid tank
  - good for builing medium to small parts with high resolution details
  - wide array of material types, but no single resin can match all the desible mechanical properties of ABS
  - system is high maintenance, proprietary resin carttridges are more expensive

### 3D Printing Structured Materials 
<center>
  <img src="imgs/infill_patterns.jpeg" width="25%">
  <font size="5">  Credit Prusa Printers Featured Print by VargaMark (https://www.prusaprinters.org/prints/62606-infill-pattern-sample-hexagon-collection)
</font>
</center>

Infilling solid volumes with a sparse support structure is a common technique used in FDM 3D printing to save material and weight without compromising strength too much

- peformed by many print preprocessors "slicers" (eg. Slic3r, Cura)

- patterns use simple geometries, are tweakable, but ultimately limited

- pores are often closed and hidden behind a skin layer

**Important Problem** SLA printers often do not support infilling in their processing pipeline, since trapped liquid resin and suction "cupping" force can ruin prints

###  The Gyroid
<center>
  <img src="imgs/schoengeometry_best_G_photo.jpg" width="25%">
  <font size="5">First plastic gyroid model, credit Alan Schoen 1968 (https://schoengeometry.com/e-tpms.html)</font>
</center>

Gyroids are triply periodic volume spanning minimal surfaces which have a cursiously simple mathematical expression: 

 $\sin(x) \cos(y)) + \sin(y) \cos(z) + \sin(z) \cos(x) = 0$
 

- discovered by NASA scientist Alan Schoen ([technical note 1970](https://ntrs.nasa.gov/citations/19700020472)), who was interested in modelling soap bubbles

- gyroid incarnations are strong because they lack sharp edges and points

- can be made with a high degree of open porosity (>90%) and there are no blind holes

**Potential Solution** gyroids won't trap liquid resin as long as pores are exposed, so we may be able to fabricate them at high resolution with SLA 3D printing!

#  Making a Gyroid: "From Math to Mesh"
Let's sprint to make a 3D printable gyroid right here in this Jupyter notebook:

In [97]:
import numpy as np    #math and array functions
from numpy import sin, cos

#create a grid of points in three dimension, split into coordinate arrays
sz = 20.0
xlim, ylim, zlim = np.vstack((3*[(-sz/2,sz/2)]))           #make a cubic span
x0,x1 = xlim;y0,y1 = ylim;z0,z1 = zlim                     #easier to work with unpacked spans
res = 256                                                  # number of points per dimension
rx,ry,rz = res*np.ones(3)                                  #copy to all dims
X,Y,Z =  np.mgrid[x0:x1:rx*1j, y0:y1:ry*1j, z0:z1:rz*1j]   #strange "imaginary" (1j) step size just means "inclusive range"

#evaluate the gyroid function on the grid
F = cos(X)*sin(Y) + cos(Y)*sin(Z) + cos(Z)*sin(X)

 #show summary data
print(F.shape,F.min(),F.max())                            

(256, 256, 256) -1.4992532595319543 1.4992532595319545


We wind up with scalar volume data defined on the 3D grid, how do we visualize it?

PyVista to the rescue! Ref: https://docs.pyvista.org/examples/00-load/create-uniform-grid.html

In [98]:
import pyvista as pv  #for visualization

# Create the spatial reference
grid = pv.UniformGrid()
# Set the grid dimensions: shape + 1 because we want to inject our values on
#   the CELL data
grid.dimensions = np.array(F.shape) + 1

# Edit the spatial reference
grid.spacing = ((x1-x0)/rx, (y1-y0)/ry,(z1-z0)/rz)  # These are the cell sizes along each axis

# Add the data values to the cell data
V = 255.0*F/F.max()                                 # Rescale to full range for better color-mapping
grid.cell_data["vol"] = V.flatten(order="F")        # Flatten the array!

# Now plot the grid!
grid.plot(volume = True)                            # Apply color map with transparency kicking in below 0

ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

That looks cool, but its not *the* Gyroid surface, it's just half of this triply periodic volume function. 

And we won't be able to 3D print a volume function!

What if we if take a thin slice of this function about 0?

In [120]:
#evaluate the gyroid function on the grid
F = cos(X)*sin(Y) + cos(Y)*sin(Z) + cos(Z)*sin(X)         #repeated for clarity
Fmax = F.max()

thresh = Fmax/10                                          #a fraction of the maximum

#threshold to get a boolean array
Fs = (F > -thresh) & (F < thresh)

#compute how much of the cubic space our thin volume occupies
vol_frac = Fs.sum()/Fs.size 

#print a summary
print(f"F max: {Fmax: .3f}")
print(f"thresh: {thresh: .4f}")
print(f"Volume fraction: {vol_frac: .3f}") 

F max:  1.499
thresh:  0.1499
Volume fraction:  0.095


Let's try to visualize what we just did...

In [122]:
#we can pack this data into the grid we already made
grid.cell_data["slice"] = 1.0*Fs.flatten(order="F")     # Flatten the array, and convert to 1 -> solid, 0 -> empty

usgrid = grid.threshold(value=0.5,scalars='slice')      # Outputs an UnstructuredGrid, a subset of the original spatial grid   

usgrid.plot(color='white')

2022-01-08 00:37:30.458 (10683.783s) [        DBB06740]       vtkThreshold.cxx:84    WARN| vtkThreshold::ThresholdByUpper was deprecated for VTK 9.1 and will be removed in a future version.


ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

Nice! But it's a bit too Minecrafty (coarsely voxelated).

And I still don't think we can 3D print an UnstructedGrid, whatever that is!

OK, let's render it as a triangulated surface mesh (what most fabrication pipelines prefer).  

Then we can smooth it out, reduce the number of triangles for a smaller model data size, and export as an STL file.  

Luckily, PyVista has us covered.

In [123]:
surfmesh = usgrid.extract_surface()
print(f"Surface mesh with square faces: {surfmesh}")
trimesh = surfmesh.triangulate()
print(f"Surface mesh with triangle faces: {trimesh}")

#do a detailed smoothing a high resolution for best results
trimesh_filtered = trimesh.smooth(n_iter=500,progress_bar=True)

#downsample to reduce number of triangles, we do it in steps with smoothing in between for a better product
decimation_factor = 0.5 #overall size reduction goal < 1.0, higher is more aggressive shrinking
decimation_steps  = 3
for i in range(decimation_steps):
    trimesh_filtered = trimesh_filtered.decimate_pro(decimation_factor,progress_bar=True)
    trimesh_filtered = trimesh_filtered.smooth(n_iter=100,progress_bar=True)
    print(f"Step {i+1}: Filtered triangle mesh: {trimesh_filtered}")
    
#removed any small floating components (possibly at corners in this case)
trimesh_filtered = trimesh_filtered.extract_largest()

Surface mesh with square faces: PolyData (0x7fa74907b2e0)
  N Cells:	2038968
  N Points:	2038484
  X Bounds:	0.000e+00, 2.000e+01
  Y Bounds:	0.000e+00, 2.000e+01
  Z Bounds:	0.000e+00, 2.000e+01
  N Arrays:	4

Surface mesh with triangle faces: PolyData (0x7fa747a7c8e0)
  N Cells:	4077936
  N Points:	2038484
  X Bounds:	0.000e+00, 2.000e+01
  Y Bounds:	0.000e+00, 2.000e+01
  Z Bounds:	0.000e+00, 2.000e+01
  N Arrays:	4



Smoothing Mesh: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████[00:22<00:00]
Decimating Mesh: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████[00:13<00:00]
Smoothing Mesh: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████[00:03<00:00]


Step 1: Filtered triangle mesh: PolyData (0x7fa747adde80)
  N Cells:	2038968
  N Points:	1019000
  X Bounds:	8.242e-03, 1.999e+01
  Y Bounds:	8.150e-03, 1.999e+01
  Z Bounds:	8.259e-03, 1.999e+01
  N Arrays:	1



Decimating Mesh: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████[00:07<00:00]
Smoothing Mesh: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████[00:02<00:00]


Step 2: Filtered triangle mesh: PolyData (0x7fa747ac4be0)
  N Cells:	1019484
  N Points:	509258
  X Bounds:	1.078e-02, 1.999e+01
  Y Bounds:	1.057e-02, 1.999e+01
  Z Bounds:	1.084e-02, 1.999e+01
  N Arrays:	1



Decimating Mesh: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████[00:03<00:00]
Smoothing Mesh: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████[00:01<00:00]

Step 3: Filtered triangle mesh: PolyData (0x7fa747a7cd00)
  N Cells:	509742
  N Points:	254387
  X Bounds:	1.428e-02, 1.999e+01
  Y Bounds:	1.427e-02, 1.998e+01
  Z Bounds:	1.433e-02, 1.999e+01
  N Arrays:	1



Let's visualize and export the final product!

In [125]:
trimesh_filtered.plot(color='white')
trimesh_filtered.save("gyroid.stl")

ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

## What is CAD?
> It takes an idea, and makes 3D stuff <br>
> Computer. Assisted. Design

They Might Be Giants "Computer Assisted Design" *Here Comes Science*

In [126]:
import IPython.display as ipd
ipd.Audio('audio/TMBG_CAD_clip_short.wav') # load a local WAV file

What kind of assistance do we get?

What drives the design process?

## To code, or not to code?
<table width="100%"><tr>
  <td>
    <figure>
      <img src="imgs/fusion360_screenshot.jpeg" width="100%">
      <figcaption><center>Autodesk Fusion 360</center></figcaption>
    </figure>
  </td> 
  <td>
    <figure>
      <img src="imgs/openSCAD_screenshot_rocket.png" width="100%">
      <figcaption><center>OpenSCAD</center></figcaption>
    </figure>
  </td> 
  <td>
    <figure>
      <img src="imgs/voxelCAD_screenshot_gyroid_sphere_octant.png" width="100%">
      <figcaption><center>VoxelCAD in Python</center></figcaption>
    </figure>
  </td> 
</tr></table>

In this talk, we will consider three examples of distinct CAD approaches as described by their primary interface:

- Graphical Application Suite: AutoDesk Fusion 360, a propreitary professional application suite with a complex graphical user interface

- Domain Specific Programming: OpenSCAD, a unique language for solid geometry construction with a simple open-source application for rendering parts (export as STL meshes)

- General Purpose Programming: Python, along with a set of 3rd party libraries can be wrangled for our structured material design purposes
  - We have started to roll out developments as an open source package, VoxelCAD

## Not to code
<center>
      <img src="imgs/fusion360_screenshot.jpeg" width="33%">
      <figcaption><center>Autodesk Fusion 360</center></figcaption>
</center>

### Graphical interface

*Advantages*
  - used more often by professionals and taught to mechanical engineers
  - powerful software suites for product design, testing, and manufacturing applications
  - no programming experience necessary
  - often can be extended by code in embedded scripting language, but this is typically a rare use case

## Not to code
<center>
    <img src="imgs/fusion360_screenshot.jpeg" width=33%>
    <div class="caption">Autodesk Fusion 360</div>
</center>

### Graphical interface

*Drawbacks*
- modular design is often challenging to setup and limited in scope
- the need to interact directly (point and click) with a complicated parts tree eventually taxes design process as complexity builds
- interfaces are quirky, complicated, and often unintuitive
    - too many menus and toolbar buttons, yet magical key-strokes are still needed to be productive
- often a part's history is inaccesable, especially if design is imported from a non-native file format
    - it can be hard to revert previous operations if you have gone on to make further modifications
- often have limited OS compatibility (except open source tools)
- the most popular propreitary software suites can be prohibitively expensive for non-educational use cases in small research group and commercial settings
      

## To code
<center>
    <img src="imgs/openSCAD_screenshot_rocket.png" width=33%>
    <div class="caption">OpenSCAD</div>
</center>

### Domain Specific Programming: OpenSCAD
*Advantages*
- encourages modular design using geometric principles
- easier to create and modify iterative structures, like arrays of variable objects and fractals
- tweaking a parametric design is often simple
- for programmers it can be a better "brain fit"
  - no need to memorize a dozen keystrokes and 100 nested menu and toolbar options
  - no strange issues with how a part was made/imported
    - the *code embodies the complete history and generates the data (mesh)*
  - refactoring can be productive
- open source and free tools, user community contributed libraries
- use your favorite version control system

## To code
<center>
    <img src="imgs/openSCAD_screenshot_rocket.png" width=33%>
    <div class="caption">OpenSCAD</div>
</center>

### Domain Specific Programming: OpenSCAD
*Drawbacks*
- less popular than GUI CAD, not usually taught in engineering coursework
- certain types of design can be more difficult, like free-form curved surfaces, filleted edges
- limited options for importing legacy parts files
- the implementation has efficiency issues with operations on complex parts
- code can be intimidating for those with less programming and math experience

## To code, more generally
<center>
    <img src="imgs/voxelCAD_screenshot_gyroid_sphere_octant.png" width=33%>
    <div class="caption">VoxelCAD in Python</div>
</center>

### General Purpose Programming: Python
*Advantages*
- many of the same coding benefits apply
- huge ecosysten of libraries, including math and science tools
- can solve design problems in new ways by composing tools from other domains
- can approximate domain specific languages with flexible Python syntax... or do it better!

## To code, more generally
<center>
    <img src="imgs/voxelCAD_screenshot_gyroid_sphere_octant.png" width=33%>
    <div class="caption">VoxelCAD in Python</div>
</center>

### General Purpose Programming: Python
*Drawbacks*
- again, certain design cases are just hard to do programmatically
- have to build up a tool chain from components
- some common CAD operations can be challenging to develop from scratch (Boolean operations on meshes, 3D rotations)